## Introduction to RLVR and GRPO Foundations

### Reinforcement Learning from Verifiable Rewards (RLVR)
In reasoning tasks like mathematics or programming, we often have access to **verifiable outcomes** (e.g., the final answer to a math problem or the output of a code snippet). Standard RLHF (Reinforcement Learning from Human Feedback) relies on reward models trained on human preferences, which can be subjective or prone to reward hacking. RLVR instead uses deterministic reward functions that check model outputs against ground truth, providing a clean signal for reinforcement.

### The GRPO Algorithm
**Group Relative Policy Optimization (GRPO)**, introduced by DeepSeek in the DeepSeek-V3 and R1 series, simplifies the RL pipeline. Unlike PPO (Proximal Policy Optimization), which requires a separate **Critic** (Value) network to estimate the baseline, GRPO computes the advantage directly by comparing a group of outputs generated from the same prompt.

#### Key Advantages:
- **No Critic Network**: Reduces VRAM requirements by avoiding the need to load a secondary large model for value estimation.
- **Group Normalization**: Advantage is calculated by normalizing the rewards within a group of $G$ outputs: $A_i = \frac{r_i - mean(r)}{std(r)}$.
- **Efficiency**: Significantly faster and more stable for tasks with sparse or discrete verifiable rewards.

#### The Objective Function
The GRPO objective function is defined as:

$$J_{GRPO}(\theta) = E[q \sim P(Q), \{o_i\}_{i=1}^G \sim \pi_{\theta}(O|q)] \left[ \frac{1}{G} \sum_{i=1}^G \min \left( \frac{\pi_{\theta}(o_i|q)}{\pi_{old}(o_i|q)} A_i, \text{clip} \left( \frac{\pi_{\theta}(o_i|q)}{\pi_{old}(o_i|q)}, 1-\epsilon, 1+\epsilon \right) A_i \right) - \beta D_{KL}(\pi_{\theta} || \pi_{ref}) \right]$$

Where:
- $G$ is the group size.
- $A_i$ is the relative advantage within the group.
- $D_{KL}$ is the KL divergence penalty to keep the model from drifting too far from the base (reference) model.

## Environment Setup and Library Deep Dive

### Subtask:
Prepare the Python environment and explain the core libraries used for RLVR with GRPO.


**Reasoning**:
The first step is to install the dependencies required for the GRPO implementation and import the relevant libraries.



In [1]:
# 1. Install dependencies using the '!' prefix for shell commands
!pip install -q trl peft accelerate bitsandbytes datasets transformers

# 2. Python imports
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import GRPOTrainer, GRPOConfig
from datasets import load_dataset

print("Libraries installed and imported successfully.")
print(f"PyTorch version: {torch.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 11.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.5 MB/s eta 0:00:00:00:0100:01
Libraries installed and imported successfully.
PyTorch version: 2.10.0+cu128


## Part 3: The GRPO Algorithm: Step-by-Step Derivation

### 1. Why this topic exists
Traditional Reinforcement Learning from Human Feedback (RLHF) uses **PPO**, which requires four models: a Policy, a Reference model, a Reward model, and a **Critic (Value Network)**. For large-scale reasoning models, keeping a Critic model (usually the same size as the Policy) in memory is extremely expensive. Furthermore, PPO's stability depends heavily on the accuracy of this Critic, which is notoriously difficult to train on sparse reasoning rewards.

### 2. Problem it solves
GRPO solves the **resource-intensity** and **training instability** of PPO. It eliminates the need for a Critic model by using **group-relative** statistics to estimate the advantage signal, drastically reducing VRAM usage and simplifying the optimization landscape.

### 3. Intuition
Imagine you are solving a math problem. Instead of a Critic guessing how 'good' your partial progress is, you generate **8 different solutions** (a group) to the same problem. You then check which solutions are correct using a verifiable reward (e.g., a regex check on the final answer).

If 2 are correct and 6 are wrong, the 2 correct ones are 'better than average' for this specific prompt. This **relative** quality within the group becomes your training signal.

### 4. Mathematical Explanation

#### The Advantage $A_i$
In PPO, $A_t = R_t - V(s_t)$. In GRPO, for a group of outputs $\{o_1, o_2, ..., o_G\}$ generated from prompt $q$, the advantage for output $o_i$ is computed as:

$$A_i = \frac{r_i - \text{mean}(\{r_1, ..., r_G\})}{\text{std}(\{r_1, ..., r_G\})}$$

This is **Group Normalization**. It ensures that the gradient updates are driven by how much better an output is compared to other samples from the same policy.

#### The Objective Function
We maximize the clipped surrogate objective, similar to PPO, but with a group-averaged KL penalty:

$$\mathcal{L}_{GRPO}(\theta) = \frac{1}{G} \sum_{i=1}^G \left[ \min \left( \frac{\pi_\theta(o_i|q)}{\pi_{old}(o_i|q)} A_i, \text{clip}\left(\frac{\pi_\theta(o_i|q)}{\pi_{old}(o_i|q)}, 1-\epsilon, 1+\epsilon\right) A_i \right) - \beta D_{KL}(\pi_\theta || \pi_{ref}) \right]$$

Where:
- $\frac{\pi_\theta}{\pi_{old}}$ is the probability ratio (importance sampling).
- $D_{KL}$ prevents the policy $\pi_\theta$ from collapsing or diverging too far from the reference model $\pi_{ref}$.

### 5. The Algorithm
1.  **Sample**: For each prompt $q$ in the batch, sample a group of $G$ responses from the current policy $\pi_\theta$.
2.  **Score**: Apply the **Verifiable Reward Function** to each response to get rewards $\{r_1, ..., r_G\}$.
3.  **Normalize**: Calculate group-relative advantages $A_i$.
4.  **Update**: Perform gradient ascent on the GRPO objective.
5.  **Iterate**: Repeat until the verifiable reward plateau or the KL divergence reaches a threshold.

---
### Summary
GRPO shifts the baseline from a learned Value Network to a dynamic group mean. This is particularly effective for **Verifiable Rewards** where the outcome is binary (Correct/Incorrect).

**Interview Question**: *Why does GRPO perform better than PPO in VRAM-constrained environments?*
**Answer**: It eliminates the Critic network, which usually consumes 25-50% of the total VRAM in a standard RLHF setup.

**Research Note**: Group size $G$ is a critical hyperparameter. Too small ($G < 4$) leads to high variance in advantage estimation; too large increases compute cost per step.

**Industry Tip**: In production, use $G=8$ or $G=16$ as a starting point for reasoning tasks.

**Common Mistake**: Forgetting to detach the Reference model, leading to accidental backpropagation through two models.

## Part 4: Policy and Reference Model Configuration

### 1. Why this topic exists
To perform RL, we need a **Policy** (the model we train) and a **Reference Model** (the frozen anchor). The Reference model is critical for calculating the KL divergence, ensuring our training doesn't result in 'mode collapse' or 'gibberish' that happens to trick the reward function.

### 2. Problem it solves
Full fine-tuning of a 7B+ parameter model during RL is computationally prohibitive for most labs. **QLoRA** allows us to train the reasoning capabilities of the model by only updating a tiny fraction (~1-2%) of the weights while the base weights remain frozen in 4-bit precision.

### 3. Hyperparameter Deep Dive

| Hyperparameter | Value | Reasoning |
| :--- | :--- | :--- |
| `learning_rate` | `5e-6` | RL is sensitive; we use a much smaller rate than SFT to avoid catastrophic forgetting. |
| `num_generations` ($G$) | `8` | The group size for GRPO. Higher values provide better advantage estimates but cost more compute. |
| `max_prompt_length` | `512` | Limits the input context to save VRAM. |
| `max_completion_length` | `1024` | Reasoning (CoT) requires space to 'think'. |
| `beta` ($\beta$) | `0.01` | The KL penalty coefficient. Controls how strictly we stay near the reference model. |
| `temperature` | `0.9` | Encourages diversity within the group $G$, which is vital for seeing both correct and incorrect answers. |

### 4. Implementation

In [ ]:
# Define the model ID
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Configuration for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

# 2. LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. GRPO Training Configuration
# Fixed: Added generation_batch_size to ensure it is divisible by num_generations
training_args = GRPOConfig(
    output_dir="./grpo-reasoning-model",
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    logging_steps=1,
    max_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=8,
    generation_batch_size=8,
    max_completion_length=512,
    temperature=0.9,
    beta=0.01,
    weight_decay=0.1,
    report_to="none"
)

print("Model and Trainer configurations fixed.")

# 4. Initialize Model and Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

print("Model loaded with QLoRA adapters.")

### Summary
We are using **Qwen-2.5-1.5B** as our base. It is highly optimized for reasoning tasks. We've configured **LoRA** to target the MLP and Attention layers, and set the **GRPO Group Size to 8**.

**Interview Question**: *Why do we use a high temperature (0.9) during GRPO training?*
**Answer**: If temperature is too low, all $G$ samples will be identical. Without variation in the group, we cannot calculate a meaningful standard deviation or mean for the rewards, and the advantage signal $A_i$ becomes zero.

**Research Note**: `beta` is often tuned dynamically. If KL divergence stays too low, the model isn't learning; if it's too high, the model is diverging.

**Common Mistake**: Setting `per_device_train_batch_size` too high. Remember that memory usage scales with `batch_size * num_generations * max_completion_length`.

## Part 5: Verifiable Reward Function and Data Pipeline

### 1. Why this topic exists
In RLVR, the 'Reward Model' is replaced by code. This eliminates the 'black box' nature of a neural reward model. For math and code, the truth is objective; if the model says $2+2=5$, it gets a 0, regardless of how 'confident' or 'polite' it was.

### 2. Problem it solves
It solves **Reward Hacking**. Standard reward models can be 'tricked' by models that use certain keywords or high-entropy tokens that the reward model associates with high quality. A verifiable Python function cannot be tricked by tone or length—only by the correctness of the result.

### 3. Intuition
We define multiple reward functions. The `GRPOTrainer` will sum these rewards for every completion.
- **Accuracy Reward**: Binary (1.0 for correct, 0.0 for wrong).
- **Format Reward**: Encourages the model to follow a specific structure (e.g., XML tags), which is essential for parsing the answer during inference.

### 4. Implementation

In [ ]:
import re
from datasets import load_dataset

def accuracy_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    Verifies the final numerical answer against the ground truth.
    """
    rewards = []
    for completion, gt_answer in zip(completions, answer):
        content = completion[0]["content"] if isinstance(completion, list) else completion
        match = re.search(r"<answer>(.*?)</answer>", content, re.DOTALL)
        if match:
            extracted = match.group(1).strip()
            rewards.append(1.0 if extracted == gt_answer else 0.0)
        else:
            rewards.append(0.0)
    return rewards

def format_reward_func(prompts, completions, **kwargs) -> list[float]:
    """
    Rewards the model for following the <thought>...<answer> format.
    """
    rewards = []
    pattern = r"^<thought>.*?</thought>\s*<answer>.*?</answer>$"
    for completion in completions:
        content = completion[0]["content"] if isinstance(completion, list) else completion
        if re.match(pattern, content, re.DOTALL):
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards

# Data Pipeline Setup with Train/Validation Split
def prepare_dataset():
    # Load full training set and split it into train and validation
    raw_dataset = load_dataset("openai/gsm8k", "main", split="train[:120]")
    dataset_split = raw_dataset.train_test_split(test_size=0.2, seed=42)

    def transform(example):
        return {
            "prompt": [{"role": "user", "content": f"Solve this: {example['question']} Finish your response with <answer>your_numerical_answer</answer>."}],
            "answer": example['answer'].split('#### ')[-1].strip()
        }

    train_ds = dataset_split['train'].map(transform)
    eval_ds = dataset_split['test'].map(transform)
    return train_ds, eval_ds

train_dataset, eval_dataset = prepare_dataset()
print(f"Training samples: {len(train_dataset)}, Validation samples: {len(eval_dataset)}")

### Summary
We've defined two reward functions: one for hard accuracy and one for soft formatting. The `GRPOTrainer` combines these into a single scalar per completion.

**Interview Question**: *Why do we use multiple reward functions instead of one big one?*
**Answer**: Modularity. It allows us to weight formatting vs. accuracy separately and monitor metrics for each independently during training.

**Research Note**: In the DeepSeek-R1 paper, they found that purely verifiable rewards (like the ones we just wrote) are sufficient to spark 'emergent reasoning' (Chain of Thought) without any explicit formatting instructions.

**Common Mistake**: Using overly strict regex. If the model adds a space or a dollar sign inside the `<answer>` tag, a simple string match might fail. In production, use more robust numerical parsers.

## Part 6: Training Loop Execution

### 1. Why this topic exists
This is where the mathematical theory meets execution. The `GRPOTrainer` handles the complexity of sampling $G$ completions for each prompt, passing them through the reward functions, and calculating the relative advantage for the policy update.

### 2. Problem it solves
It automates the iterative RL process. Without a specialized trainer like `GRPOTrainer` from the `trl` library, we would have to manually manage the importance sampling ratios, clipping, and KL penalty calculations across the group.

### 3. Implementation

In [ ]:
from trl import GRPOTrainer

# Re-initialize the GRPOTrainer with train and eval datasets
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[accuracy_reward_func, format_reward_func],
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

print("Starting GRPO Training with validation tracking...")
trainer.train()

# Save the final adapter
trainer.save_model("./grpo-qwen-gsm8k")
print("Training complete and model saved.")

### Summary
The training loop is now optimizing the model based on the verifiable rewards.

**Interview Question**: *What happens if the model's reward increases but the responses become repetitive?*
**Answer**: This is a sign of mode collapse. To fix it, you might increase the `beta` (KL penalty) to stay closer to the diverse base model or increase the `temperature` during sampling to force more exploration.

**Research Note**: In RLVR, you often see a 'spike' in loss early on as the model transitions from its SFT behavior to exploring high-reward reasoning paths.

**Common Mistake**: Forgetting to save the model. Since we are using LoRA, `trainer.save_model` only saves the small adapter weights, not the full 1.5B parameters, making it very fast.

## Part 7: Training Dynamics and Evaluation

### 1. Why this topic exists
RL is notoriously opaque. Monitoring the scalar reward isn't enough; we need to see the 'Reasoning Length' (the number of tokens inside `<thought>` tags). In RLVR, a common phenomenon is that reasoning length increases as the model discovers that longer thought chains lead to more accurate answers.

### 2. Implementation: Quick Evaluation

In [6]:
import matplotlib.pyplot as plt

# 1. Post-training Inference Test
def test_inference(test_prompt):
    # Apply chat template and return tensors
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": test_prompt}],
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # Fix: Unpack inputs dictionary into generate to provide input_ids and attention_mask correctly
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

sample_q = "If I have 100 apples and buy 20 more, then eat 50, how many do I have?"
print(f"Test Inference:\n{test_inference(sample_q)}")

NameError: name 'tokenizer' is not defined

### 3. Visualizing Training Dynamics

We extract the `log_history` from the trainer to visualize the reward growth and KL divergence. This is the 'EKG' of your RL process.

In [5]:
import pandas as pd
import seaborn as sns

# Extract logs
history = trainer.state.log_history
df_logs = pd.DataFrame([log for log in history if 'loss' in log])

# Create plots
fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot Loss
sns.lineplot(data=df_logs, x='step', y='loss', ax=ax1, color='b', label='Policy Loss')
ax1.set_ylabel('Loss')
ax1.set_title('GRPO Training Dynamics')

# Plot Reward on secondary axis if available
if 'reward' in df_logs.columns:
    ax2 = ax1.twinx()
    sns.lineplot(data=df_logs, x='step', y='reward', ax=ax2, color='g', label='Mean Reward')
    ax2.set_ylabel('Reward')

plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

NameError: name 'trainer' is not defined

## Final Synthesis & Research Notes

### Industry Tips for Deployment
1.  **Iterative SFT-then-RL**: Always perform a Supervised Fine-Tuning (SFT) step on high-quality reasoning traces before starting GRPO. This gives the model a 'starting map' of the desired format.
2.  **Reward Shaping**: If the model fails to find any correct answers initially (sparse reward), add a small reward for partial matches or correct thought-tag usage to 'warm up' the policy.
3.  **VLLM Integration**: For production-scale RLVR, use `vLLM` as the generation engine inside `GRPOTrainer` to speed up the sampling phase by 3-5x.

### Critical Thinking / Interview Questions
- **Q**: How does the KL penalty $\beta$ affect the 'creativity' of the reasoning?
- **A**: A high $\beta$ forces the model to stay identical to the base model, stifling exploration. A low $\beta$ allows the model to explore new reasoning paths but risks 'mode collapse' where the model outputs gibberish that somehow satisfies a buggy reward function.